# Bölüm 5c — Brickset'ten Toplu Retail Price Çekimi

**Amaç:** Faz 5c kapsamında, BrickEconomy'ye ek/tamamlayıcı bir kaynak olarak
[Brickset](https://brickset.com) API v3'ünden (`getSets`) tüm analiz-hazır
setler için retail price verisini **toplu** şekilde çekmek. BrickEconomy'nin
aksine Brickset ücretsiz ve günlük kota (100 `getSets` çağrısı) çok daha
yüksek olduğu için, tüm setleri **yıl bazlı sorgularla tek oturumda**
çekebiliyoruz.

**Strateji (önceki keşif adımında netleştirildi):**
- Kimlik doğrulama: `apiKey` + boş `userHash` (public set verisi için hash
  gerekmiyor)
- Sorgu birimi: yıl (`sets_clean.csv`'deki 75 farklı yıl, 1949–2025)
- Sayfalama: `pageSize=2000` + `matches` kontrolü (taşarsa `pageNumber` ile
  devam)
- Rate limit: istekler arası ~15 saniye bekleme (dakikada 4 istek)
- Günlük kota güvenliği: her yıldan önce `getKeyUsageStats` ile kontrol,
  90/100'e ulaşılırsa **dur**, kalan yılları ertesi güne bırak
- Kesintiye dayanıklılık (resumable): her yılın ham sonucu
  `data/raw/brickset/{year}.json` olarak kaydedilir; dosya zaten varsa o yıl
  tekrar çekilmez

Yardımcı fonksiyonlar [`src/brickset_api.py`](../src/brickset_api.py) içinde.

In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT / "src"))

import pandas as pd
from dotenv import load_dotenv
import os

load_dotenv(PROJECT_ROOT / ".env")
api_key = os.environ.get("BRICKSET_API_KEY")
assert api_key, "BRICKSET_API_KEY .env içinde bulunamadı"

from brickset_api import fetch_all_years, extract_price_row

RAW_DIR = PROJECT_ROOT / "data/raw/brickset"
PROCESSED_DIR = PROJECT_ROOT / "data/processed"
print("Proje kökü:", PROJECT_ROOT)

Proje kökü: /Users/didemarslan/Downloads/brick_by_brick_data


## 1) Çekilecek yılları belirle

In [2]:
sets_clean = pd.read_csv(PROCESSED_DIR / "sets_clean.csv")
ready = sets_clean[sets_clean["is_analysis_ready"] == True]
years = sorted(int(y) for y in ready["year"].unique())

print(f"is_analysis_ready==True set sayısı: {len(ready)}")
print(f"Farklı yıl sayısı: {len(years)} ({years[0]}-{years[-1]})")
print("En yoğun 5 yıl:")
print(ready["year"].value_counts().sort_values(ascending=False).head(5))

is_analysis_ready==True set sayısı: 18190
Farklı yıl sayısı: 75 (1949-2025)
En yoğun 5 yıl:
year
2025    855
2024    777
2021    721
2023    710
2022    684
Name: count, dtype: int64


## 2) Toplu çekim (resumable)

Bu hücre her yıl için `data/raw/brickset/{year}.json` dosyasını kontrol eder;
varsa atlar, yoksa günlük kota kontrolüyle birlikte çeker. Not defteri daha
önce çalıştırılmışsa (ör. arka planda script ile), bu hücrenin çalıştırılması
sadece eksik yılları tamamlar — API çağrısı yapmadan hızlıca biter.

In [3]:
result = fetch_all_years(api_key, years, RAW_DIR)
print()
print("=== Özet ===")
print(f"Bu çalıştırmada yeni çekilen yıl sayısı: {len(result['fetched'])}")
print(f"Zaten önbellekte olup atlanan yıl sayısı: {len(result['skipped_cached'])}")
print(f"Durdurulan yıl (varsa): {result['stopped_at']}")
print(f"Son bilinen günlük kullanım: {result['last_known_usage']}")

cached_years = sorted(int(p.stem) for p in RAW_DIR.glob("*.json"))
missing_years = sorted(set(years) - set(cached_years))
print(f"\nToplam önbellekte bulunan yıl dosyası: {len(cached_years)}/{len(years)}")
if missing_years:
    print(f"Eksik kalan yıllar (ertesi gün devam edilecek): {missing_years}")
else:
    print("Tüm yıllar başarıyla çekildi.")


=== Özet ===
Bu çalıştırmada yeni çekilen yıl sayısı: 0
Zaten önbellekte olup atlanan yıl sayısı: 75
Durdurulan yıl (varsa): None
Son bilinen günlük kullanım: None

Toplam önbellekte bulunan yıl dosyası: 75/75
Tüm yıllar başarıyla çekildi.


## 3) JSON'ları birleştir → `brickset_prices.csv`

In [4]:
import json

rows = []
for path in sorted(RAW_DIR.glob("*.json")):
    year_data = json.loads(path.read_text())
    for set_obj in year_data.get("sets", []):
        rows.append(extract_price_row(set_obj))

brickset_prices = pd.DataFrame(rows)
before = len(brickset_prices)
brickset_prices = brickset_prices.dropna(subset=["brickset_set_num"])
brickset_prices = brickset_prices.drop_duplicates(subset=["brickset_set_num"], keep="first")
after = len(brickset_prices)
print(f"Ham satır: {before}, brickset_set_num'suz/tekrarlı düşülen: {before - after}, final: {after}")

out_path = PROCESSED_DIR / "brickset_prices.csv"
brickset_prices.to_csv(out_path, index=False)
print(f"Kaydedildi: {out_path}")
brickset_prices.head()

Ham satır: 22311, brickset_set_num'suz/tekrarlı düşülen: 0, final: 22311
Kaydedildi: /Users/didemarslan/Downloads/brick_by_brick_data/data/processed/brickset_prices.csv


,brickset_set_num,retail_price_us,retail_price_uk,retail_price_ca,retail_price_de,date_first_available,date_last_available
0,700-12,NaN,NaN,NaN,NaN,NaN,NaN
1,700_1_1-1,NaN,NaN,NaN,NaN,NaN,NaN
2,700_1_2-1,NaN,NaN,NaN,NaN,NaN,NaN
3,700_A-1,NaN,NaN,NaN,NaN,NaN,NaN
4,700_B_1-1,NaN,NaN,NaN,NaN,NaN,NaN


## 4) `sets_clean.csv` ile LEFT JOIN testi (kalıcı değil — sadece eşleşme oranını ölçmek için)

`sets_clean.csv`'deki `set_num` ile `brickset_prices.csv`'deki
`brickset_set_num` üzerinden test amaçlı bir LEFT JOIN. **Bu birleştirme
`sets_clean.csv`'ye kalıcı olarak yazılmıyor** — sadece Bölüm 2 (fiyat
tahmini regresyonu) için gerçekte kaç setin retail price verisiyle
kullanılabilir olacağını görmek içindir.

In [5]:
test_merge = sets_clean.merge(
    brickset_prices, left_on="set_num", right_on="brickset_set_num", how="left"
)

total = len(test_merge)
matched = test_merge["retail_price_us"].notna().sum()
unmatched = total - matched

print(f"Toplam set (sets_clean.csv): {total}")
print(f"Retail price ile eşleşen: {matched} ({matched/total:.1%})")
print(f"Eşleşmeyen (retail price'sız kalan): {unmatched} ({unmatched/total:.1%})")

ready_total = len(sets_clean[sets_clean['is_analysis_ready'] == True])
ready_matched = test_merge[test_merge['is_analysis_ready'] == True]['retail_price_us'].notna().sum()
print()
print(f"Sadece is_analysis_ready==True içinde: {ready_matched}/{ready_total} ({ready_matched/ready_total:.1%}) eşleşti")
print("(Bölüm 2 regresyonu için gerçek kullanılabilir örneklem büyüklüğü budur.)")

Toplam set (sets_clean.csv): 18899
Retail price ile eşleşen: 6620 (35.0%)
Eşleşmeyen (retail price'sız kalan): 12279 (65.0%)

Sadece is_analysis_ready==True içinde: 6614/18190 (36.4%) eşleşti
(Bölüm 2 regresyonu için gerçek kullanılabilir örneklem büyüklüğü budur.)


## 5) Not: Brickset yıl toplamları neden bizimkinden yüksek çıkıyor?

Keşif adımında gördüğümüz gibi, Brickset'in bir yıl için döndürdüğü toplam
set sayısı (`matches`), `sets_clean.csv`'deki aynı yılın set sayısından
genelde daha yüksek (ör. 2020: Brickset 860 vs. bizim csv 678, ~%27 fark).

**Neden:** Brickset'in veritabanı sadece "normal" LEGO setlerini değil, aynı
zamanda bizim Faz 4'te (`sets_clean.csv` temizliğinde) zaten filtrelediğimiz
**merch/gear/promosyon kategorilerini** de (anahtarlıklar, kitaplar,
sadece-minifigür paketleri, "gear" ürünleri vb.) içeriyor.

**Bunun eşleştirmeyi etkilememesinin nedeni:** Yukarıdaki LEFT JOIN,
**`sets_clean.csv`'yi temel alıyor** (sol taraf) — yani sadece bizim zaten
sahip olduğumuz/filtrelediğimiz setler için Brickset'te bir eşleşme arıyoruz.
Brickset tarafındaki "fazladan" merch/gear satırları LEFT JOIN'de hiç
görünmüyor (sağ tarafta kalıp solda karşılığı olmadığı için otomatik
düşüyorlar). Dolayısıyla yıl toplamlarındaki bu fark, yukarıdaki eşleşme
oranını hiçbir şekilde bozmuyor — sadece Brickset'in daha geniş bir
kategori yelpazesini kapsadığını gösteriyor.